In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.tsa.stattools import ccf

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 300})

print("=======================================================")
print(" SCRIPT 4: ANÁLISIS VISUAL (INTERDEPENDENCIA Y CCF)")
print("=======================================================")

df_ml = pd.read_parquet('data/ml_ready/dataset_ozono_predictivo.parquet')

# --- 1. DISTRIBUCIONES FÍSICO-ESTADÍSTICAS ---
print("Generando Gráficos de Ajuste de Distribuciones...")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

shape_log, loc_log, scale_log = stats.lognorm.fit(df_ml['PM2.5'])
x_pm25 = np.linspace(df_ml['PM2.5'].min(), df_ml['PM2.5'].max(), 100)
axes[0].hist(df_ml['PM2.5'], bins=50, density=True, alpha=0.5, color='orange')
axes[0].plot(x_pm25, stats.lognorm.pdf(x_pm25, shape_log, loc_log, scale_log), 'r-', lw=2, label='Lognormal Fit')
axes[0].set_title('Distribución Asimétrica de $PM_{2.5}$')
axes[0].legend()

shape_gam, loc_gam, scale_gam = stats.gamma.fit(df_ml['PM10'])
x_pm10 = np.linspace(df_ml['PM10'].min(), df_ml['PM10'].max(), 100)
axes[1].hist(df_ml['PM10'], bins=50, density=True, alpha=0.5, color='gray')
axes[1].plot(x_pm10, stats.gamma.pdf(x_pm10, shape_gam, loc_gam, scale_gam), 'b-', lw=2, label='Gamma Fit')
axes[1].set_title('Distribución de Eventos Extremos $PM_{10}$')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/figures/Etapa3_01_Distribuciones.png')
plt.close()

# --- 2. CORRELACIÓN CRUZADA (CCF) ---
print("Ejecutando Correlación Cruzada (CCF)...")
lags_to_plot = 48
ccf_pm25_o3 = ccf(df_ml['PM2.5_12h'], df_ml['O3_8h'], adjusted=False)[:lags_to_plot]
ccf_sr_o3 = ccf(df_ml['SR'], df_ml['O3_8h'], adjusted=False)[:lags_to_plot]

# IMPRESIÓN TRANSPARENTE EN CONSOLA
min_lag_pm = np.argmin(ccf_pm25_o3)
max_lag_sr = np.argmax(ccf_sr_o3)
print(f"-> Máximo bloqueo radiativo detectado en: Lag {min_lag_pm} horas.")
print(f"-> Pico de acumulación fotoquímica detectado en: Lag {max_lag_sr} horas.")

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
lags = np.arange(0, lags_to_plot)

axes[0].vlines(lags, [0], ccf_pm25_o3, color='crimson', lw=3)
axes[0].axhline(0, color='black', lw=1)
axes[0].set_title('Correlación Cruzada: $PM_{2.5}$ (NowCast) liderando a $O_3$ (8h)')

axes[1].vlines(lags, [0], ccf_sr_o3, color='orange', lw=3)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Correlación Cruzada: Radiación Solar liderando a $O_3$ (8h)')

plt.tight_layout()
plt.savefig('results/figures/Etapa3_02_CCF.png')
plt.close()
print("✅ ¡Análisis visual completado y exportado!")

 SCRIPT 4: ANÁLISIS VISUAL (INTERDEPENDENCIA Y CCF)
Generando Gráficos de Ajuste de Distribuciones...


d:\Codingggg\Notas\Multivariados\MA2003B_Eq1\.venv\Lib\site-packages\scipy\stats\_continuous_distns.py:6986: RuntimeWarning: divide by zero encountered in log
  return np.sum((1 + np.log(shifted/scale)/shape**2)/shifted)


Ejecutando Correlación Cruzada (CCF)...
-> Máximo bloqueo radiativo detectado en: Lag 3 horas.
-> Pico de acumulación fotoquímica detectado en: Lag 17 horas.
✅ ¡Análisis visual completado y exportado!
